# 18 — Preliminary rollout vs damped-SW (paired CI budgets)

Priority-2 plumbing: load notebook 12 ``outputs/sw_alpha_bo.json`` metadata when present,
then run a **paired** ``rollout_eval`` grid at fixed α=0.9 with CI-sized episode budgets
(``n_burn=1``, ``n_score=3``, rollout ``H=4``, ``n_paths=2``, ``radius=1``).

``SMOKE=False`` keeps both arms and two validation seeds; use ``BATCH_MODE="local"``
without a Modal account.

In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Literal

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
for _candidate in [REPO_ROOT, *REPO_ROOT.parents]:
    if (_candidate / "src" / "blueberries_voi").is_dir():
        REPO_ROOT = _candidate
        break

_wheel_dir = REPO_ROOT / "dist" / "wheel"
_gsin_bin = REPO_ROOT / "target" / "release" / "examples" / "gsin_upc_diag"
os.environ.setdefault("BLUEBERRIES_VOI_BACKEND", "rust")
if _wheel_dir.is_dir():
    os.environ["BLUEBERRIES_VOI_WHEEL"] = str(_wheel_dir)
if _gsin_bin.is_file():
    os.environ["GSIN_UPC_DIAG_BIN"] = str(_gsin_bin)

from blueberries_voi.experiments.modal_dispatch import run_batch
from blueberries_voi.sim.alpha_tune import DEFAULT_CI_ALPHAS

BATCH_MODE: Literal["modal", "local"] = "modal"
SMOKE = False

BO_JSON = REPO_ROOT / "outputs" / "sw_alpha_bo.json"
bo_meta: dict = {}
if BO_JSON.is_file():
    bo_meta = json.loads(BO_JSON.read_text(encoding="utf-8"))
    print(f"Loaded {BO_JSON.name}: arm={bo_meta.get('arm')} full_run={bo_meta.get('full_run')}")
else:
    print(f"No {BO_JSON.name}; using fixed CI defaults below")

VAL_SEEDS = tuple(bo_meta.get("val_seeds", (42, 7))[:2]) if bo_meta else (42, 7)
ALPHA = 0.9 if 0.9 in DEFAULT_CI_ALPHAS else float(DEFAULT_CI_ALPHAS[-1])
N_BURN, N_SCORE = 1, 3
ROLLOUT_H, N_PATHS, RADIUS = 4, 2, 1
RHO = float(bo_meta.get("best_rho_profit_soo", 0.8))

Loaded sw_alpha_bo.json: arm=sw full_run=False


## Paired rollout_eval (sw + rollout)

In [2]:
rows = run_batch(
    "rollout_eval",
    BATCH_MODE,
    smoke=SMOKE,
    seeds=VAL_SEEDS,
    arms=("sw", "rollout"),
    alphas=(ALPHA,),
    rho=RHO,
    n_burn=N_BURN,
    n_score=N_SCORE,
    rollout_h=ROLLOUT_H,
    n_rollout_paths=N_PATHS,
    candidate_case_radius=RADIUS,
)
df = pd.DataFrame(rows)
summary = df.groupby("arm_id")[["profit", "waste", "stockout"]].mean()
summary

[2026-08-24T17:05:41Z] nb13 grid: 1/4 complete


[2026-08-24T17:05:41Z] nb13 grid: 2/4 complete


[2026-08-24T17:05:46Z] nb13 grid: 3/4 complete


[2026-08-24T17:05:47Z] nb13 grid: 4/4 complete


[2026-08-24T17:05:47Z] modal batch collected 4 shards


,profit,waste,stockout
arm_id,,,
rollout,-137.75,0.5,59.0
sw,-137.75,0.5,59.0


## Paired Δprofit (rollout − sw)

In [3]:
sw = df[df["arm_id"] == "sw"].set_index("seed")["profit"]
ro = df[df["arm_id"] == "rollout"].set_index("seed")["profit"]
delta = (ro - sw).dropna()
mean_delta = float(delta.mean())
sem = float(delta.std(ddof=1) / np.sqrt(len(delta))) if len(delta) > 1 else float("nan")
print(
    f"paired Δprofit (rollout - sw): mean={mean_delta:.2f} "
    f"SEM={sem:.2f} n={len(delta)} α={ALPHA} rho={RHO}"
)

paired Δprofit (rollout - sw): mean=0.00 SEM=0.00 n=2 α=0.9 rho=0.6975510115735233
